In [ ]:
import torch
import torch.nn as nn
import numpy as np
from xgboost import XGBClassifier
import os
import pandas as pd
from glob import glob
import rasterio as rio
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from src.mslandcover.models import HRNetSegmentationModel
from src.mslandcover.config import HRNET_BASE_CONFIG

In [135]:
target_paths = glob('data/png_images/batch_0/target/*.tif')
input_paths = [x.replace('target', 'input_tif') for x in target_paths]

img_paths_df = pd.DataFrame({'input': input_paths, 'target': target_paths})
img_paths_df['X'] = img_paths_df['input'].apply(lambda x: rio.open(x).read())
img_paths_df['y'] = img_paths_df['target'].apply(lambda x: rio.open(x).read())

In [3]:
hrnet_w18_config = {
    'STAGE1': {
        'NUM_MODULES': 1,
        'NUM_BRANCHES': 1,
        'BLOCK': 'BOTTLENECK',
        'NUM_BLOCKS': [4],
        'NUM_CHANNELS': [64],
        'FUSE_METHOD': 'SUM',
    },
    'STAGE2': {
        'NUM_MODULES': 1,
        'NUM_BRANCHES': 2,
        'BLOCK': 'BASIC',
        'NUM_BLOCKS': [4, 4],
        'NUM_CHANNELS': [18, 36],
        'FUSE_METHOD': 'SUM',
    },
    'STAGE3': {
        'NUM_MODULES': 4,
        'NUM_BRANCHES': 3,
        'BLOCK': 'BASIC',
        'NUM_BLOCKS': [4, 4, 4],
        'NUM_CHANNELS': [18, 36, 72],
        'FUSE_METHOD': 'SUM',
    },
    'STAGE4': {
        'NUM_MODULES': 3,
        'NUM_BRANCHES': 4,
        'BLOCK': 'BASIC',
        'NUM_BLOCKS': [4, 4, 4, 4],
        'NUM_CHANNELS': [18, 36, 72, 144],
        'FUSE_METHOD': 'SUM',
    },
    'IMAGE_DECODER': {
        'NUM_BLOCKS': 2, # number of blocks per decoder layer - 2 is the default for simplicity
    },
    'SIMCLR_PROJECTION_HEAD': {
        'NUM_HIDDENS': 1, # number of hidden layers to use in the projection head
        'EMBED_DIM': 128,
    }
}

In [137]:
X_batch = np.stack(img_paths_df['X'].values)
X_batch = torch.from_numpy(X_batch).float()

model = HRNetSegmentationModel(hrnet_w18_config)
# remove incre_modules in encoder
# model.encoder.incre_modules = nn.Identity()

# remove decoder
model.decoder = nn.Identity()
model.img_decoder_activation = nn.Identity()

# remove projection head
model.projection_head = nn.Identity()
model.load_state_dict(torch.load('./weights/hrnet_w18/hsv_simclr_old.pth'), strict=False)
    
print(model)
# model.load_state_dict(torch.load('./weights/hrnet_w18/hsv_simclr_old.pth', weights_only=True), strict=False)

C:\Users\dh2306\AppData\Local\Temp\ipykernel_53056\2172633861.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('./weights/hrnet_w18/hsv_

HRNetSegmentationModel(
  (encoder): HighResolutionNet(
    (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, 

In [138]:
model.eval()
model = model.to('cuda')

# print(X_batch.shape)

with torch.no_grad():
    X_batch = X_batch.to('cuda')
    y_pred = model(X_batch)[0] # only return the first output - projection head is missing
    
    y_pred = nn.functional.interpolate(y_pred, scale_factor=4, mode='bilinear', align_corners=True)

print(y_pred.shape)
img_paths_df['z'] = list(y_pred.cpu().numpy())

torch.Size([55, 270, 256, 256])


In [114]:
print(img_paths_df['z'].values.shape)
print(len(img_paths_df['X']))
print(len(img_paths_df['y']))  

(55,)
55
55


In [139]:
n_features = img_paths_df['z'].values[0].shape[0]
print(n_features)

270


In [189]:
# add hand-crafted features
def caculate_ndvi(X):
    # nir band is 0
    # red band is 1
    X = X.astype(np.float32)
    return (X[0] - X[1]) / (X[0] + X[1])

def caculate_ndwi(X):
    # nir band is 0
    # green band is 2
    X = X.astype(np.float32)
    return (X[2] - X[0]) / (X[2] + X[0])

def caclulate_gndvi(X):
    # nir band is 0
    # green band is 2
    X = X.astype(np.float32)
    return (X[0] - X[2]) / (X[0] + X[2])



img_paths_df['ndvi'] = img_paths_df['X'].apply(caculate_ndvi)
img_paths_df['ndwi'] = img_paths_df['X'].apply(caculate_ndwi)
img_paths_df['gndvi'] = img_paths_df['X'].apply(caclulate_gndvi)

# append the hand-crafted features to the model output
agg_features = np.stack(img_paths_df['z'].values)

print(np.stack(img_paths_df['X'].values)[:,0][:, None].shape)
print(np.stack(img_paths_df['ndvi'].values)[:, None].shape)

# print(X.shape)
agg_features = np.concatenate([
    agg_features, 
    np.stack(img_paths_df['ndvi'].values)[:, None], 
    np.stack(img_paths_df['ndwi'].values)[:, None], 
    np.stack(img_paths_df['gndvi'].values)[:, None], 
    np.stack(img_paths_df['X'].values)[:,0][:, None],
    np.stack(img_paths_df['X'].values)[:,1][:, None],
    np.stack(img_paths_df['X'].values)[:,2][:, None]],
axis=1)
n_features = agg_features.shape[1]

img_paths_df['agg_features'] = list(agg_features)

C:\Users\dh2306\AppData\Local\Temp\ipykernel_53056\3432741601.py:6: RuntimeWarning: invalid value encountered in divide
  return (X[0] - X[1]) / (X[0] + X[1])
C:\Users\dh2306\AppData\Local\Temp\ipykernel_53056\3432741601.py:12: RuntimeWarning: invalid value encountered in divide
  return (X[2] - X[0]) / (X[2] + X[0])
C:\Users\dh2306\AppData\Local\Temp\ipykernel_53056\3432741601.py:18: RuntimeWarning: invalid value encountered in divide
  return (X[0] - X[2]) / (X[0] + X[2])


(55, 1, 256, 256)
(55, 1, 256, 256)


In [190]:
train_df, test_df = train_test_split(img_paths_df, test_size=0.2)

offset = (256 - 192) // 2 # only keep the center 192x192 pixels due to edge effects

# print(np.stack(.shape)

X_train = np.stack(train_df['agg_features'])[:, :, offset:-offset, offset:-offset].transpose(0, 2, 3, 1).reshape(-1, n_features)
y_train = np.stack(train_df['y'].values)[:, :, offset:-offset, offset:-offset].flatten()

X_test = np.stack(test_df['agg_features'])[:, :, offset:-offset, offset:-offset].transpose(0, 2, 3, 1).reshape(-1, n_features)
y_test = np.stack(test_df['y'].values)[:, :, offset:-offset, offset:-offset].flatten()

# X_test = np.stack([z.transpose(1, 2, 0) for z in test_df['z'].values])
# X_test = X_test.reshape(-1, 720)
# y_test = np.stack([y.transpose(1, 2, 0) for y in test_df['y'].values])
# y_test = y_test.reshape(-1, 1)

In [212]:
np.stack(test_df['y'].values)[:, :, offset:-offset, offset:-offset].shape

(11, 1, 192, 192)

In [96]:
print(len(X_train), len(y_train))

1622016 1622016


In [191]:
# print distributio of classes
print(np.unique(y_train, return_counts=True))

# remove 0 class from the data
X_train = X_train[y_train != 0]
y_train = y_train[y_train != 0] - 1

X_test = X_test[y_test != 0]
y_test = y_test[y_test != 0] - 1

from imblearn.over_sampling import SMOTE

oversample = SMOTE()
X_train, y_train = oversample.fit_resample(X_train, y_train)

# print(np.unique(y_train, return_counts=True))



(array([0, 1, 2, 3, 4, 5, 6, 7, 8], dtype=uint8), array([ 73963,  66675,  22623,  55516, 115075, 917502, 260410,  42681,
        67571]))


In [192]:
model = XGBClassifier(n_estimators=100, n_jobs=-1)

model.fit(X_train, y_train)

print(model.score(X_test, y_test))

y_preds = model.predict(X_test)
print(classification_report(y_test, y_preds))

0.7675424249633243
              precision    recall  f1-score   support

           0       0.97      0.80      0.88      1737
           1       0.45      0.59      0.51      2509
           2       0.68      0.63      0.65      9040
           3       0.38      0.31      0.34     12388
           4       0.88      0.97      0.92    262540
           5       0.46      0.72      0.56     51132
           6       0.00      0.00      0.00     51561
           7       0.58      0.59      0.59     12629

    accuracy                           0.77    403536
   macro avg       0.55      0.57      0.56    403536
weighted avg       0.68      0.77      0.72    403536



In [193]:
fis = model.feature_importances_
sorted_f1s = np.argsort(fis)[::-1]
print(sorted_f1s)

[272 270 213  85 274 271 275  40  32 220 178 247 151 123 164  27 145  22
 126 200 273 194 139  66  84 223 160  89 149 235 211 265 156 228 114 207
  46 208 249 174 267 179  54 250 137 175  43 221 132  94 237 150 186 184
 128 130 170 225 234 269 183 254 100 138 144 140 209 176 163  76 142  73
 227 143 152 199 240 233 258 189 268  13  38 222 248 241 263 219 187  39
 232  34  11 242  41 202 162 260 121  93 116 124 135 177 226 141  88 230
 109 166 133 167 239 196 257 127  95 206  74 262 205 154 204 136 236 192
 104 218 193 168  25 251  98 161  65 190 182 216  62 215  24 146  47  86
 158 243 148 180 155 229  91  82 105 259  52  49 119 102 231 191  58 246
  68 171  15 210  51  99  45  79  92  35  64 115 120 224  31 172  97  77
  83  44  60  50 125 134 106 112  72  55  90 129 157  57  80 203  20  16
 169  28  69 103   0   3  26 117  17   8 153  53   1  56 245 173  19  42
  12  23  36 253  14  71  10 201 256  87 244 264 110 217  67   2 131   5
 266  18 159   7   6   4   9 261 147 165  81 212 21

In [194]:
X_train_spectral = X_train[:, n_features-6:]
X_test_spectral = X_test[:, n_features-6:]

model_spectral = XGBClassifier(n_estimators=100, n_jobs=-1)

model_spectral.fit(X_train_spectral, y_train)

print(model_spectral.score(X_test_spectral, y_test))

y_preds_spectral = model_spectral.predict(X_test_spectral)

print(classification_report(y_test, y_preds_spectral))

0.595919570992427
              precision    recall  f1-score   support

           0       0.89      0.84      0.86      1737
           1       0.35      0.71      0.47      2509
           2       0.66      0.54      0.59      9040
           3       0.36      0.66      0.47     12388
           4       0.89      0.68      0.77    262540
           5       0.41      0.64      0.50     51132
           6       0.10      0.10      0.10     51561
           7       0.26      0.69      0.38     12629

    accuracy                           0.60    403536
   macro avg       0.49      0.61      0.52    403536
weighted avg       0.68      0.60      0.62    403536



In [ ]:
# visualize predictions from each model

import matplotlib.pyplot as plt

y_test_sample = y_test.reshape(-1, 
y_preds_sample = y_preds.reshape(-1, 192, 192)[0]
y_preds_spectral_sample = y_preds_spectral.reshape(-1, 192, 192)[0]

fig, axs = plt.subplots(1, 3, figsize=(15, 5))

axs[0].imshow(y_test_sample)
axs[0].set_title('Ground Truth')

axs[1].imshow(y_preds_sample)
axs[1].set_title('Deep + Spectral + Hand-crafted Features')

axs[2].imshow(y_preds_spectral_sample)
axs[2].set_title('Spectral + Hand-Crafted Features Only')


ValueError: cannot reshape array of size 403536 into shape (64,64)

In [213]:
y_test_reshaped = y_test.reshape(11, 1, 192, 192)


ValueError: cannot reshape array of size 403536 into shape (11,1,192,192)

635.2448346897438